[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/01_Multimodal_Foundations/02_modality_encoders/02_modality_encoders.ipynb)

# 02. Modality Encoders: Image & Text

**This notebook covers:**
- How ViT (Vision Transformer) encodes images — with visualization
- How text encoders (BERT-style) produce embeddings
- Building both from scratch and using pretrained versions
- Visualizing what each encoder learns

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/01_Multimodal_Foundations/02_modality_encoders")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

## Part 1: Vision Transformer (ViT) from Scratch

**Key idea:** Split an image into patches, treat each patch as a "token", then apply a transformer.

```
Image (224×224) → 16×16 patches → 196 patch tokens → Transformer → [CLS] embedding
```

![ViT Architecture — Dosovitskiy et al. (2020)](../../assets/paper_figures/vit_architecture.png)
*Source: Dosovitskiy et al. (2020) — An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*

![Vision Transformer (ViT) — Dosovitskiy et al. (2020)](../assets/paper_figure_vit.png)

*Source: Dosovitskiy et al. (2020) — "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale" — [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)*

### Self-Attention — The Core Mechanism

ViT applies the same self-attention you know from text transformers, but over **image patch tokens**:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

**Why divide by $\sqrt{d_k}$?**

Without scaling, dot products $q \cdot k$ for dimension $d_k$ have variance $\approx d_k$ (each term contributes ~1 to the sum). For $d_k = 64$, dot products cluster around ±64, pushing softmax into **saturation** (values near 0 or 1) where gradients are near-zero.

With scaling by $\sqrt{d_k}$:
- Dot product variance → **≈ 1**
- Softmax inputs stay in a range where gradients flow well
- Training is stable from the start

This scaling factor is critical — removing it causes training to diverge in practice.

### Multi-Head Attention

Instead of one attention function, we run $H$ parallel "heads" and concatenate:

$$\text{MultiHead}(X) = \text{Concat}(h_1, \ldots, h_H) \, W^O$$

where each head computes:

$$h_i = \text{Attention}(X W_i^Q,\; X W_i^K,\; X W_i^V)$$

**Parameter count per layer:** $4 \times d \times d$ for the Q, K, V, O projection matrices (each is $d \times d$).

**Why multiple heads?** Each head can learn different relationship patterns — one head might attend to local patch neighbors, another to globally salient regions. With $H=12$ heads and $d=768$, each head operates in $d_k = 768/12 = 64$ dimensions.

### Example 1: Self-Attention — Complete Step-by-Step with Real Numbers

Let's trace self-attention by hand for **3 tokens** with **$d_k = 4$** (tiny for clarity):

**Input tokens** (after embedding + position):

$$X = \begin{bmatrix} 1.0 & 0.5 & -0.3 & 0.8 \\ 0.2 & -0.7 & 1.2 & 0.1 \\ -0.5 & 0.9 & 0.4 & -0.6 \end{bmatrix}$$

Row 0 = "cat", Row 1 = "sits", Row 2 = "mat"

In [ ]:
# ============================================================
#  Example 1: Self-Attention — Step by Step with Real Numbers
# ============================================================
import torch
import torch.nn.functional as F
import numpy as np

print("=" * 70)
print("  SELF-ATTENTION: Complete Step-by-Step Numerical Trace")
print("=" * 70)

# Input: 3 tokens, d=4
X = torch.tensor([
    [1.0, 0.5, -0.3, 0.8],   # "cat"
    [0.2, -0.7, 1.2, 0.1],   # "sits"
    [-0.5, 0.9, 0.4, -0.6]   # "mat"
])
token_names = ["cat", "sits", "mat"]
d_k = X.shape[1]  # 4

# Weight matrices (small for hand-tracing)
torch.manual_seed(42)
W_Q = torch.randn(4, 4) * 0.5
W_K = torch.randn(4, 4) * 0.5
W_V = torch.randn(4, 4) * 0.5

print(f"\nInput X (3 tokens × d={d_k}):")
for i, name in enumerate(token_names):
    print(f"  {name:5s}: [{X[i,0]:6.2f}, {X[i,1]:6.2f}, {X[i,2]:6.2f}, {X[i,3]:6.2f}]")

# Step 1: Compute Q, K, V
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print(f"\nStep 1: Q = X @ W_Q")
for i, name in enumerate(token_names):
    print(f"  Q[{name:5s}] = [{Q[i,0]:6.3f}, {Q[i,1]:6.3f}, {Q[i,2]:6.3f}, {Q[i,3]:6.3f}]")

print(f"\n  K = X @ W_K")
for i, name in enumerate(token_names):
    print(f"  K[{name:5s}] = [{K[i,0]:6.3f}, {K[i,1]:6.3f}, {K[i,2]:6.3f}, {K[i,3]:6.3f}]")

print(f"\n  V = X @ W_V")
for i, name in enumerate(token_names):
    print(f"  V[{name:5s}] = [{V[i,0]:6.3f}, {V[i,1]:6.3f}, {V[i,2]:6.3f}, {V[i,3]:6.3f}]")

# Step 2: Q @ K^T
scores = Q @ K.T
print(f"\nStep 2: Raw scores = Q @ K^T  (3×3 matrix)")
print(f"  {'':7s}", end="")
for name in token_names:
    print(f"  {name:>7s}", end="")
print()
for i, name in enumerate(token_names):
    print(f"  {name:7s}", end="")
    for j in range(3):
        print(f"  {scores[i,j]:7.3f}", end="")
    print()

# Step 3: Scale by sqrt(d_k)
scale = d_k ** 0.5
scaled = scores / scale
print(f"\nStep 3: Scaled scores = scores / √{d_k} = scores / {scale:.2f}")
print(f"  {'':7s}", end="")
for name in token_names:
    print(f"  {name:>7s}", end="")
print()
for i, name in enumerate(token_names):
    print(f"  {name:7s}", end="")
    for j in range(3):
        print(f"  {scaled[i,j]:7.3f}", end="")
    print()

# Step 4: Softmax
attn = torch.softmax(scaled, dim=-1)
print(f"\nStep 4: Attention weights = softmax(scaled scores)")
print(f"  {'':7s}", end="")
for name in token_names:
    print(f"  {name:>7s}", end="")
print("    sum")
for i, name in enumerate(token_names):
    print(f"  {name:7s}", end="")
    for j in range(3):
        print(f"  {attn[i,j]:7.4f}", end="")
    print(f"  = {attn[i].sum():.4f}")

# Step 5: Weighted sum of values
output = attn @ V
print(f"\nStep 5: Output = Attention @ V")
print(f"  Interpretation: each token is a weighted average of all Value vectors")
for i, name in enumerate(token_names):
    print(f"\n  output[{name}] = ", end="")
    parts = []
    for j, n2 in enumerate(token_names):
        parts.append(f"{attn[i,j]:.3f}×V[{n2}]")
    print(" + ".join(parts))
    print(f"          = [{output[i,0]:6.3f}, {output[i,1]:6.3f}, {output[i,2]:6.3f}, {output[i,3]:6.3f}]")

print(f"\n{'='*70}")
print(f"  KEY INSIGHT: 'cat' attends most to '{token_names[attn[0].argmax()]}' (weight={attn[0].max():.3f})")
print(f"  'sits' attends most to '{token_names[attn[1].argmax()]}' (weight={attn[1].max():.3f})")
print(f"  'mat'  attends most to '{token_names[attn[2].argmax()]}' (weight={attn[2].max():.3f})")
print(f"{'='*70}")

In [ ]:
# Visualize the attention pattern
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Attention weights heatmap
ax = axes[0]
im = ax.imshow(attn.numpy(), cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(token_names)
ax.set_yticklabels(token_names)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{attn[i,j]:.3f}', ha='center', va='center', fontsize=12,
                color='white' if attn[i,j] > 0.5 else 'black')
ax.set_title('Attention Weights\n(who attends to whom)', fontsize=13, fontweight='bold')
ax.set_xlabel('Keys (attended to)')
ax.set_ylabel('Queries (attending)')
plt.colorbar(im, ax=ax)

# Raw scores vs scaled scores
ax = axes[1]
x = np.arange(9)
raw = scores.flatten().numpy()
scl = scaled.flatten().numpy()
width = 0.35
ax.bar(x - width/2, raw, width, label='Raw QK^T', color='#E74C3C', alpha=0.7)
ax.bar(x + width/2, scl, width, label=f'Scaled (÷√{d_k})', color='#3498DB', alpha=0.7)
labels = [f'{n1}→{n2}' for n1 in token_names for n2 in token_names]
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Score')
ax.set_title(f'Effect of √d_k Scaling\n(prevents softmax saturation)', fontsize=13, fontweight='bold')
ax.legend()

# Softmax temperature effect
ax = axes[2]
test_logits = torch.tensor([2.0, 1.0, 0.5])
temps = [0.1, 0.5, 1.0, 2.0, 5.0]
for temp in temps:
    probs = torch.softmax(test_logits / temp, dim=0).numpy()
    ax.plot([0, 1, 2], probs, 'o-', label=f'T={temp}', linewidth=2, markersize=8)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['High', 'Medium', 'Low'])
ax.set_ylabel('Probability')
ax.set_title('Softmax Temperature Effect\n(lower T = sharper distribution)', fontsize=13, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../assets/attention_numerical.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 1: Visualize patch extraction

def visualize_patches(image_tensor, patch_size=4):
    """Show how an image is split into patches."""
    C, H, W = image_tensor.shape
    n_patches_h = H // patch_size
    n_patches_w = W // patch_size
    n_patches = n_patches_h * n_patches_w

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Original image with grid
    ax = axes[0]
    img_np = image_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    ax.imshow(img_np)
    for i in range(0, H + 1, patch_size):
        ax.axhline(y=i, color='red', linewidth=1, alpha=0.7)
    for j in range(0, W + 1, patch_size):
        ax.axvline(x=j, color='red', linewidth=1, alpha=0.7)
    ax.set_title(f'Image with {patch_size}×{patch_size} patch grid\n({n_patches} patches total)', fontsize=12)
    ax.axis('off')

    # Show individual patches
    ax = axes[1]
    show_n = min(n_patches, 16)
    grid_size = int(np.ceil(np.sqrt(show_n)))
    
    patches = image_tensor.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    patches = patches.contiguous().view(C, -1, patch_size, patch_size)
    patches = patches.permute(1, 2, 3, 0)  # [N, H, W, C]

    combined = np.zeros((grid_size * (patch_size + 1), grid_size * (patch_size + 1), 3))
    for idx in range(show_n):
        r, c = idx // grid_size, idx % grid_size
        patch = patches[idx].numpy()
        patch = (patch - patch.min()) / (patch.max() - patch.min() + 1e-8)
        y_start = r * (patch_size + 1)
        x_start = c * (patch_size + 1)
        combined[y_start:y_start+patch_size, x_start:x_start+patch_size] = patch

    ax.imshow(combined)
    ax.set_title(f'First {show_n} patches (extracted)', fontsize=12)
    ax.axis('off')

    plt.tight_layout()
    return fig

# Create a colorful test image
test_image = torch.zeros(3, 16, 16)
test_image[0, :8, :8] = 1.0    # Red top-left
test_image[1, :8, 8:] = 1.0    # Green top-right
test_image[2, 8:, :8] = 1.0    # Blue bottom-left
test_image[:, 8:, 8:] = 0.8     # White bottom-right
test_image += torch.randn_like(test_image) * 0.1

fig = visualize_patches(test_image, patch_size=4)
plt.show()

In [ ]:
# STEP 2: Build ViT from scratch

class PatchEmbedding(nn.Module):
    """Convert image into patch embeddings using a convolution."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, 
                             kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: [B, C, H, W] -> [B, embed_dim, H/P, W/P] -> [B, N, embed_dim]
        x = self.proj(x)                    # [B, embed_dim, grid_h, grid_w]
        x = x.flatten(2).transpose(1, 2)    # [B, N_patches, embed_dim]
        return x


class ViTFromScratch(nn.Module):
    """Minimal Vision Transformer."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 embed_dim=128, n_heads=4, n_layers=4, n_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True,
            dropout=0.1, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)

    def forward(self, x, return_features=False):
        B = x.shape[0]
        x = self.patch_embed(x)                                # [B, N, D]
        cls = self.cls_token.expand(B, -1, -1)                 # [B, 1, D]
        x = torch.cat([cls, x], dim=1)                         # [B, N+1, D]
        x = x + self.pos_embed                                 # Add position info
        x = self.transformer(x)                                # [B, N+1, D]
        x = self.norm(x)
        cls_output = x[:, 0]                                   # [CLS] token
        
        if return_features:
            return cls_output
        return self.head(cls_output)


vit = ViTFromScratch(img_size=32, patch_size=4, embed_dim=128, n_layers=4)
count_parameters(vit)

# Test forward pass
dummy = torch.randn(2, 3, 32, 32)
out = vit(dummy)
print(f"\nInput shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")

features = vit(dummy, return_features=True)
print(f"Feature shape: {features.shape}  ← This goes to the shared space")

### ViT-Base Parameter Breakdown

Understanding where parameters live helps with memory planning:

| Component | Calculation | Parameters |
|-----------|-------------|------------|
| **Patch embedding** | $3 \times 16 \times 16 \times 768$ | **590K** |
| **Transformer (12 layers)** | $12 \times (4 \times 768^2 + 2 \times 768 \times 3072)$ | **~85M** |
| **Position embeddings** | $197 \times 768$ (196 patches + CLS) | **151K** |
| **Total** | | **≈ 86M** |

The **transformer blocks dominate** (~99% of params). Each layer contains:
- Self-attention: $4 \times 768 \times 768 = 2.36\text{M}$ (Q, K, V, O projections)
- FFN: $2 \times 768 \times 3072 = 4.72\text{M}$ (expand 4×, then project back)

This is why **LoRA** targets attention projections — that's where the bulk of learnable capacity lives.

### Example 2: ViT-Base — Memory and Compute Budget

How much memory and compute does ViT-Base need? Let's calculate:

**Memory for weights (inference):**
- ViT-Base: 86M params × 4 bytes (FP32) = **344 MB**
- ViT-Base: 86M params × 2 bytes (FP16) = **172 MB**

**Memory for activations (batch=32, seq_len=197, d=768):**
- Input activations: 32 × 197 × 768 × 4 bytes = **18.5 MB** per layer
- With 12 layers (for backprop): **222 MB**
- Attention matrices: 32 × 12 × 197 × 197 × 4 bytes = **58 MB** per layer → **~700 MB** total

**FLOPs per forward pass (single image):**
- Self-attention: $4 \times 197 \times 768^2 + 2 \times 197^2 \times 768 = 464\text{M} + 59.7\text{M}$ per layer
- FFN: $2 \times 197 \times 768 \times 3072 = 929\text{M}$ per layer  
- Total: $12 \times (464 + 59.7 + 929) \approx$ **17.4 GFLOPs** per image

**Throughput estimate** (A100 @ 312 TFLOPS FP16):
- $312 \times 10^{12} / (17.4 \times 10^9) \approx$ **17,931 images/sec** theoretical max
- Real-world (~40% utilization): **~7,000 images/sec**

In [ ]:
# ============================================================
#  Example 2: ViT-Base Memory and Compute Budget Calculator
# ============================================================

def compute_vit_budget(img_size=224, patch_size=16, d_model=768, 
                       n_layers=12, n_heads=12, ffn_mult=4, batch_size=32):
    """Calculate memory and compute requirements for ViT."""
    n_patches = (img_size // patch_size) ** 2
    seq_len = n_patches + 1  # +1 for CLS token
    d_ff = d_model * ffn_mult
    d_k = d_model // n_heads
    
    print(f"{'='*60}")
    print(f"  ViT Budget Calculator")
    print(f"{'='*60}")
    print(f"  Image: {img_size}×{img_size}, Patch: {patch_size}×{patch_size}")
    print(f"  Patches: {n_patches} + 1 CLS = {seq_len} tokens")
    print(f"  d_model={d_model}, layers={n_layers}, heads={n_heads}")
    print(f"  d_k = {d_model}/{n_heads} = {d_k}")
    print(f"{'='*60}")
    
    # Parameter count
    patch_embed_params = 3 * patch_size * patch_size * d_model + d_model  # conv + bias
    pos_embed_params = seq_len * d_model
    cls_token_params = d_model
    
    attn_params_per_layer = 4 * d_model * d_model + 4 * d_model  # QKVO + biases
    ffn_params_per_layer = 2 * d_model * d_ff + d_model + d_ff  # two linears + biases
    norm_params_per_layer = 4 * d_model  # 2 LayerNorms × (gamma + beta)
    layer_params = attn_params_per_layer + ffn_params_per_layer + norm_params_per_layer
    
    total_params = patch_embed_params + pos_embed_params + cls_token_params + n_layers * layer_params
    
    print(f"\n📦 Parameters:")
    print(f"  Patch embedding:  {patch_embed_params:>12,}")
    print(f"  Position embed:   {pos_embed_params:>12,}")
    print(f"  CLS token:        {cls_token_params:>12,}")
    print(f"  Per layer:        {layer_params:>12,} × {n_layers} = {layer_params * n_layers:,}")
    print(f"    Attention:      {attn_params_per_layer:>12,}")
    print(f"    FFN:            {ffn_params_per_layer:>12,}")
    print(f"    LayerNorm:      {norm_params_per_layer:>12,}")
    print(f"  {'─'*40}")
    print(f"  Total:            {total_params:>12,} ({total_params/1e6:.1f}M)")
    
    # Memory
    mem_fp32 = total_params * 4 / 1e9
    mem_fp16 = total_params * 2 / 1e9
    
    act_per_layer = batch_size * seq_len * d_model * 4 / 1e9  # FP32
    attn_per_layer = batch_size * n_heads * seq_len * seq_len * 4 / 1e9
    total_act = n_layers * (act_per_layer + attn_per_layer)
    
    print(f"\n💾 Memory (batch_size={batch_size}):")
    print(f"  Weights (FP32):   {mem_fp32*1000:>8.1f} MB")
    print(f"  Weights (FP16):   {mem_fp16*1000:>8.1f} MB")
    print(f"  Activations/layer: {act_per_layer*1000:>7.1f} MB (tokens) + {attn_per_layer*1000:.1f} MB (attn)")
    print(f"  Total activations: {total_act*1000:>7.1f} MB (for backprop, all layers)")
    print(f"  Training total:    {(mem_fp32 + total_act)*1000:>7.1f} MB (FP32, no optimizer)")
    
    # FLOPs
    attn_flops = (4 * seq_len * d_model * d_model + 2 * seq_len * seq_len * d_model) * n_layers
    ffn_flops = (2 * seq_len * d_model * d_ff) * n_layers
    total_flops = attn_flops + ffn_flops
    
    print(f"\n⚡ Compute (single image forward):")
    print(f"  Attention FLOPs:  {attn_flops/1e9:>8.2f} GFLOPs")
    print(f"  FFN FLOPs:        {ffn_flops/1e9:>8.2f} GFLOPs")
    print(f"  Total FLOPs:      {total_flops/1e9:>8.2f} GFLOPs")
    
    # Throughput estimates
    a100_tflops = 312  # FP16 Tensor Cores
    t4_tflops = 65     # FP16
    
    print(f"\n🚀 Estimated Throughput (images/sec):")
    for gpu_name, tflops in [("A100-80GB", 312), ("V100-16GB", 125), ("T4-16GB", 65), ("RTX 3090", 142)]:
        theoretical = tflops * 1e12 / total_flops
        practical = theoretical * 0.4  # ~40% utilization
        print(f"  {gpu_name:>12s}: {practical:>8,.0f} img/s (at 40% util)")
    
    return total_params, total_flops

# Run for common ViT variants
print("\n" + "="*60)
print("  Comparison Across ViT Variants")
print("="*60)

variants = {
    "ViT-Tiny":  dict(d_model=192,  n_layers=12, n_heads=3,  ffn_mult=4),
    "ViT-Small": dict(d_model=384,  n_layers=12, n_heads=6,  ffn_mult=4),
    "ViT-Base":  dict(d_model=768,  n_layers=12, n_heads=12, ffn_mult=4),
    "ViT-Large": dict(d_model=1024, n_layers=24, n_heads=16, ffn_mult=4),
}

print(f"\n{'Variant':<12} {'Params':>10} {'GFLOPs':>10} {'FP16 MB':>10}")
print("─" * 44)
for name, cfg in variants.items():
    params, flops = compute_vit_budget(**cfg, batch_size=1)
    print(f"{name:<12} {params/1e6:>9.1f}M {flops/1e9:>9.2f} {params*2/1e6:>9.1f}")
    print()  # spacing

In [ ]:
# STEP 3: Visualize what ViT sees at each stage

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('ViT Processing Pipeline (Visual)', fontsize=16, fontweight='bold')

img = torch.randn(1, 3, 32, 32)

# Stage 1: Original image
ax = axes[0, 0]
ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1).numpy())
ax.set_title('1. Input Image\n(3 × 32 × 32)')
ax.axis('off')

# Stage 2: Patch embeddings
ax = axes[0, 1]
patch_emb = vit.patch_embed(img)  # [1, 64, 128]
ax.imshow(patch_emb[0].detach().numpy()[:, :32], aspect='auto', cmap='viridis')
ax.set_title(f'2. Patch Embeddings\n{patch_emb.shape[1]} patches × {patch_emb.shape[2]} dim')
ax.set_xlabel('Embedding dimensions (first 32)')
ax.set_ylabel('Patch index')

# Stage 3: Position embeddings
ax = axes[0, 2]
pos = vit.pos_embed[0].detach().numpy()
ax.imshow(pos[:, :32], aspect='auto', cmap='coolwarm')
ax.set_title(f'3. Position Embeddings\n{pos.shape[0]} positions (CLS + patches)')
ax.set_xlabel('Embedding dimensions (first 32)')
ax.set_ylabel('Position')

# Stage 4: After transformer (token representations)
ax = axes[1, 0]
with torch.no_grad():
    B = 1
    x = vit.patch_embed(img)
    cls = vit.cls_token.expand(B, -1, -1)
    x = torch.cat([cls, x], dim=1) + vit.pos_embed
    x = vit.transformer(x)

ax.imshow(x[0].numpy()[:, :32], aspect='auto', cmap='viridis')
ax.set_title(f'4. After Transformer\n{x.shape[1]} tokens × {x.shape[2]} dim')
ax.set_xlabel('Embedding dimensions (first 32)')
ax.set_ylabel('Token index (0=CLS)')

# Stage 5: CLS token (final feature)
ax = axes[1, 1]
cls_feat = x[0, 0].numpy()
ax.bar(range(len(cls_feat[:64])), cls_feat[:64], color='#9B59B6', alpha=0.7)
ax.set_title(f'5. [CLS] Token Vector\n(first 64 of {len(cls_feat)} dims)')
ax.set_xlabel('Dimension')
ax.set_ylabel('Value')

# Stage 6: Classification logits
ax = axes[1, 2]
logits = vit(img).detach().numpy()[0]
bars = ax.bar(range(10), logits, color='#E74C3C', alpha=0.7)
ax.set_title('6. Output Logits\n(10 classes)')
ax.set_xlabel('Class')
ax.set_ylabel('Logit value')

plt.tight_layout()
plt.savefig('../assets/vit_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Text Encoder

For multimodal models, the text encoder converts a sentence into a fixed-size vector.  
Common approach: Transformer encoder → take [CLS] token → project to shared dim.

### Positional Encoding

Transformers have no inherent notion of order — positional encodings inject **sequence position** information:

**Sinusoidal (original Transformer):**

$$PE(\text{pos}, 2i) = \sin\left(\frac{\text{pos}}{10000^{2i/d}}\right), \quad PE(\text{pos}, 2i+1) = \cos\left(\frac{\text{pos}}{10000^{2i/d}}\right)$$

**Intuition:**
- **Low-frequency** dimensions (large wavelength) → capture **long-range** position ("beginning vs end of sentence")
- **High-frequency** dimensions (small wavelength) → capture **fine-grained** position ("token 3 vs token 4")

ViT uses **learnable** positional embeddings (our `pos_embed` parameter), while BERT/GPT often use learned or sinusoidal encodings. Both approaches work; learnable embeddings are simpler and ViT's ablations show minimal difference.

### Layer Normalization

LayerNorm stabilizes training by normalizing **across features within each token**:

$$\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu$ and $\sigma$ are computed **per-token** (across the embedding dimension $d$), and $\gamma, \beta$ are learnable scale/shift parameters.

**Pre-LN vs Post-LN:**

| Variant | Placement | Used In |
|---------|-----------|---------|
| **Post-LN** | After residual: $x + \text{Sublayer}(\text{LN}(x))$ | Original Transformer (2017) |
| **Pre-LN** | Before sublayer: $x + \text{Sublayer}(\text{LN}(x))$ | ViT, GPT-2/3, modern LLMs |

Pre-LN produces **more stable gradients** in deep networks — the normalization happens before attention/FFN, preventing activation blow-up. Both ViT and our text encoder use Pre-LN (PyTorch's `TransformerEncoderLayer` defaults to Pre-LN since v1.9).

In [ ]:
class TextEncoderFromScratch(nn.Module):
    """BERT-style text encoder with visualization hooks."""
    def __init__(self, vocab_size=30000, embed_dim=128, n_heads=4,
                 n_layers=2, max_len=64):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        self.type_embed = nn.Embedding(2, embed_dim)  # segment A/B

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self._attention_weights = []

    def forward(self, input_ids, token_type_ids=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)

        x = self.token_embed(input_ids) + self.pos_embed(positions)
        if token_type_ids is not None:
            x = x + self.type_embed(token_type_ids)

        x = self.transformer(x)
        x = self.norm(x)
        return x[:, 0]  # [CLS] representation


text_enc = TextEncoderFromScratch(vocab_size=30000, embed_dim=128)
count_parameters(text_enc)

# Test
dummy_ids = torch.randint(0, 30000, (2, 16))
output = text_enc(dummy_ids)
print(f"\nInput shape:  {dummy_ids.shape}  (batch=2, seq_len=16)")
print(f"Output shape: {output.shape}  (batch=2, embed_dim=128)")

In [ ]:
# Visualize text encoding pipeline

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Text Encoding Pipeline', fontsize=16, fontweight='bold')

tokens = ['[CLS]', 'a', 'photo', 'of', 'a', 'cute', 'cat', '[SEP]', '[PAD]', '[PAD]']
dummy_ids = torch.randint(0, 100, (1, len(tokens)))

# 1. Token IDs
ax = axes[0]
ax.barh(range(len(tokens)), dummy_ids[0].numpy(), color='#3498DB', alpha=0.7)
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens)
ax.set_title('1. Token IDs')
ax.set_xlabel('ID value')
ax.invert_yaxis()

# 2. Token embeddings
ax = axes[1]
tok_emb = text_enc.token_embed(dummy_ids)[0].detach().numpy()
ax.imshow(tok_emb[:, :32], aspect='auto', cmap='RdBu_r')
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens)
ax.set_title('2. Token Embeddings')
ax.set_xlabel('Dim (first 32)')

# 3. + Position embeddings
ax = axes[2]
pos_emb = text_enc.pos_embed(torch.arange(len(tokens)))
combined = tok_emb + pos_emb.detach().numpy()
ax.imshow(combined[:, :32], aspect='auto', cmap='RdBu_r')
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens)
ax.set_title('3. Token + Position')
ax.set_xlabel('Dim (first 32)')

# 4. Output [CLS] vector
ax = axes[3]
with torch.no_grad():
    output = text_enc(dummy_ids)
ax.barh(range(32), output[0, :32].numpy(), color='#2ECC71', alpha=0.7)
ax.set_title(f'4. [CLS] Output\n(first 32 of {output.shape[1]})')
ax.set_xlabel('Value')
ax.set_ylabel('Dimension')

plt.tight_layout()
plt.savefig('../assets/text_encoding_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

### Example 3: Real-World — Feature Extraction with Pretrained Encoders

In practice, you rarely build encoders from scratch. Here's how the complete pipeline works for a **product search** application:

```
Customer photo → ViT encoder → 768-dim vector ─┐
                                                 ├─→ cosine similarity → top-K results
Product description → BERT encoder → 768-dim vector ─┘
```

**The complete flow:**
1. Customer uploads a photo of a dress
2. ViT encodes it to a 768-dimensional vector
3. All product descriptions are pre-encoded with BERT
4. Cosine similarity finds the closest matches
5. Return top-5 products

This is **exactly what** Amazon Visual Search, Pinterest Lens, and Google Lens do under the hood.

In [ ]:
# ============================================================
#  Example 3: Simulated Product Search Pipeline
# ============================================================

# Build a simple retrieval system using our encoders
vit.eval()
text_enc.eval()

# Simulate a product database
N_PRODUCTS = 100
torch.manual_seed(42)

# Product images (synthetic)
product_images = torch.randn(N_PRODUCTS, 3, 32, 32)

# Product text descriptions (synthetic token IDs)
product_texts = torch.randint(0, 30000, (N_PRODUCTS, 16))

# Assign categories
categories = ['dress', 'shoe', 'hat', 'bag', 'watch'] * 20
# Add category signal to images
for i in range(N_PRODUCTS):
    cat_id = i % 5
    product_images[i, cat_id % 3, :8, :8] += 3.0  # visual signal

print("Product Search Demo")
print("=" * 50)

# Pre-encode all products
with torch.no_grad():
    img_features = vit(product_images, return_features=True)  # [100, 128]
    img_features = img_features / img_features.norm(dim=-1, keepdim=True)
    
    txt_features = text_enc(product_texts)  # [100, 128]
    txt_features = txt_features / txt_features.norm(dim=-1, keepdim=True)

# Simulate a query: "show me a dress" (use first product as query image)
query_image = product_images[0:1]  # A "dress" image
with torch.no_grad():
    query_feat = vit(query_image, return_features=True)
    query_feat = query_feat / query_feat.norm(dim=-1, keepdim=True)

# Image-to-image retrieval
i2i_sims = (query_feat @ img_features.T).squeeze()
top5_i2i = torch.topk(i2i_sims, 6)  # top 6 (first is the query itself)

print(f"\n🔍 Query: Product #0 (category: {categories[0]})")
print(f"\nTop-5 Image-to-Image Retrieval:")
print(f"  {'Rank':>4}  {'Product':>8}  {'Category':>10}  {'Similarity':>10}")
print(f"  {'─'*36}")
for rank, (idx, sim) in enumerate(zip(top5_i2i.indices[1:], top5_i2i.values[1:])):
    match = "✅" if categories[idx] == categories[0] else "❌"
    print(f"  {rank+1:>4}  #{idx.item():>7}  {categories[idx]:>10}  {sim.item():>10.4f}  {match}")

# Cross-modal retrieval: text query → images
query_text = product_texts[0:1]  # Use text matching "dress"
with torch.no_grad():
    query_txt_feat = text_enc(query_text)
    query_txt_feat = query_txt_feat / query_txt_feat.norm(dim=-1, keepdim=True)

t2i_sims = (query_txt_feat @ img_features.T).squeeze()
top5_t2i = torch.topk(t2i_sims, 5)

print(f"\nTop-5 Text-to-Image Retrieval:")
print(f"  {'Rank':>4}  {'Product':>8}  {'Category':>10}  {'Similarity':>10}")
print(f"  {'─'*36}")
for rank, (idx, sim) in enumerate(zip(top5_t2i.indices, top5_t2i.values)):
    match = "✅" if categories[idx] == categories[0] else "❌"
    print(f"  {rank+1:>4}  #{idx.item():>7}  {categories[idx]:>10}  {sim.item():>10.4f}  {match}")

# Visualize similarity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
same_cat = [i2i_sims[i].item() for i in range(N_PRODUCTS) if categories[i] == categories[0] and i != 0]
diff_cat = [i2i_sims[i].item() for i in range(N_PRODUCTS) if categories[i] != categories[0]]
ax.hist(same_cat, bins=15, alpha=0.7, color='#2ECC71', label=f'Same category (n={len(same_cat)})')
ax.hist(diff_cat, bins=15, alpha=0.7, color='#E74C3C', label=f'Different category (n={len(diff_cat)})')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Count')
ax.set_title('Image-to-Image Similarity\n(same vs different category)', fontsize=13, fontweight='bold')
ax.legend()

ax = axes[1]
ax.scatter(range(N_PRODUCTS), i2i_sims.numpy(), 
           c=['#2ECC71' if categories[i] == categories[0] else '#E74C3C' for i in range(N_PRODUCTS)],
           alpha=0.7, s=30)
ax.set_xlabel('Product Index')
ax.set_ylabel('Similarity to Query')
ax.set_title('Retrieval Scores\n(green = same category)', fontsize=13, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../assets/product_search_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 3: Using Pretrained Encoders (Practical)

In practice, you use pretrained encoders and just add a projection head.

### GELU Activation

Modern transformers (ViT, BERT, GPT) use **GELU** instead of ReLU:

$$\text{GELU}(x) = x \cdot \Phi(x) \approx 0.5x\left(1 + \tanh\left[\sqrt{2/\pi}\,(x + 0.044715 x^3)\right]\right)$$

where $\Phi(x)$ is the standard Gaussian CDF.

**Why GELU over ReLU?**
| Property | ReLU | GELU |
|----------|------|------|
| Smoothness | Sharp corner at 0 | Smooth everywhere |
| Negative values | Hard zero | Small negative values allowed |
| Gradient flow | Dead neurons possible | No dead zone |

GELU's smooth, probabilistic gating (it can be viewed as "stochastic regularization") works better with LayerNorm and self-attention — which is why ViT, BERT, and GPT all default to GELU in their feed-forward layers.

In [ ]:
from transformers import AutoModel, AutoTokenizer, ViTModel, ViTFeatureExtractor
from torchvision import models

# Option 1: Use a pretrained ResNet as image encoder
resnet = models.resnet18(weights=None)  # no download needed for structure
resnet_features = nn.Sequential(*list(resnet.children())[:-1])  # remove classifier

dummy_img = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    feat = resnet_features(dummy_img)
print(f"ResNet18 feature shape: {feat.shape}  → flatten to {feat.flatten(1).shape}")
print(f"Then project: Linear(512, shared_dim)")

# Count params
print(f"\nResNet18 total params: {sum(p.numel() for p in resnet.parameters()):,}")

In [ ]:
# Comparison of encoder sizes (for low-compute planning)

encoder_data = {
    'Model': ['ResNet-18', 'ResNet-50', 'ViT-Tiny', 'ViT-Small', 'ViT-Base',
              'BERT-Tiny', 'BERT-Mini', 'BERT-Base', 'DistilBERT'],
    'Params (M)': [11.7, 25.6, 5.7, 22.1, 86.6,
                   4.4, 11.3, 110, 66],
    'Type': ['Vision', 'Vision', 'Vision', 'Vision', 'Vision',
             'Text', 'Text', 'Text', 'Text'],
    'Low Compute': ['Yes', 'Yes', 'Yes', 'Yes', 'Moderate',
                      'Yes', 'Yes', 'Moderate', 'Yes']
}

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#E74C3C' if t == 'Vision' else '#3498DB' for t in encoder_data['Type']]
edge_colors = ['green' if lc == 'Yes' else 'orange' for lc in encoder_data['Low Compute']]

bars = ax.barh(encoder_data['Model'], encoder_data['Params (M)'], 
               color=colors, alpha=0.7, edgecolor=edge_colors, linewidth=2)

for bar, params in zip(bars, encoder_data['Params (M)']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{params}M', va='center', fontweight='bold')

ax.set_xlabel('Parameters (Millions)', fontsize=12)
ax.set_title('Encoder Size Comparison\n(green border = low-compute friendly)', fontsize=14, fontweight='bold')

import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#E74C3C', alpha=0.7, label='Vision Encoder'),
    mpatches.Patch(color='#3498DB', alpha=0.7, label='Text Encoder'),
], fontsize=11)

plt.tight_layout()
plt.savefig('../assets/encoder_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **ViT** splits images into patches → treats them as tokens → transformer processes them
2. **Text encoder** converts tokens → embeddings → transformer → [CLS] output
3. Both produce a **fixed-size vector** that goes into the shared space
4. For low compute: use **ViT-Tiny/Small** and **DistilBERT/BERT-Mini**
5. The **projection head** maps encoder outputs to the shared dimension

---
**Next:** `03_fusion_strategies.ipynb` - How to combine modalities

---

## 📚 References & Further Reading

### Papers
- **An Image is Worth 16x16 Words: Transformers for Image Recognition (ViT)** — Dosovitskiy et al., 2020 — [arXiv:2010.11929](https://arxiv.org/abs/2010.11929) — Vision Transformer
- **BERT: Pre-training of Deep Bidirectional Transformers** — Devlin et al., 2018 — [arXiv:1810.04805](https://arxiv.org/abs/1810.04805) — Text encoder foundation
- **Attention Is All You Need** — Vaswani et al., 2017 — [arXiv:1706.03762](https://arxiv.org/abs/1706.03762) — Original Transformer
- **Deep Residual Learning (ResNet)** — He et al., 2015 — [arXiv:1512.03385](https://arxiv.org/abs/1512.03385) — CNN backbone for vision
- **DeiT: Training Data-Efficient Image Transformers** — Touvron et al., 2021 — [arXiv:2012.12877](https://arxiv.org/abs/2012.12877) — Efficient ViT training
- **BEiT: BERT Pre-Training of Image Transformers** — Bao et al., 2021 — [arXiv:2106.08254](https://arxiv.org/abs/2106.08254) — Self-supervised ViT
- **Gaussian Error Linear Units (GELUs)** — Hendrycks & Gimpel, 2016 — [arXiv:1606.08415](https://arxiv.org/abs/1606.08415)

### Blog Posts & Cheat Sheets
- 🔗 [The Illustrated ViT](https://jalammar.github.io/illustrated-vit/) — Jay Alammar — Step-by-step ViT walkthrough
- 🔗 [The Illustrated BERT](https://jalammar.github.io/illustrated-bert/) — Jay Alammar — Visual BERT explanation
- 🔗 [Vision Transformer Explained](https://theaisummer.com/vision-transformer/) — AI Summer — ViT with code
- 🔗 [Hugging Face ViT Documentation](https://huggingface.co/docs/transformers/model_doc/vit) — Pretrained ViT usage
- 🔗 [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — Harvard NLP — Line-by-line Transformer implementation
- 🔗 [timm Library (PyTorch Image Models)](https://huggingface.co/docs/timm/) — Comprehensive ViT variants